# Daraz Reviews — Data Viewer & Cleaner
Run each cell one by one with **Shift + Enter**

### Initial cleaning steps for collected data, remove unnecessary fields, duplicates, very short reviews, devnagari text etc....


In [44]:
import pandas as pd
import csv
import re

In [45]:


input_file = 'daraz_reviews_labelled.csv'
df = pd.read_csv(input_file, encoding='utf-8-sig')
print(f'✓ Loaded {len(df)} rows from {input_file}')
print(f'Columns: {list(df.columns)}')
print(f'Shape: {df.shape}')
df.head()

✓ Loaded 1602 rows from daraz_reviews_labelled.csv
Columns: ['id', 'review_text', 'source', 'product_category', 'rating', 'label']
Shape: (1602, 6)


,id,review_text,source,product_category,rating,label
0,1,saama thikai x 2 ota magayeko euta ma problem raixa....euta ramro x.,daraz.com.np,Electronics,5.0,product_feedback
1,2,Thikai xa price aanusar ko lìa hunxa,daraz.com.np,Electronics,5.0,product_feedback
2,3,"nice quality,bass good,I like it time in delivery",daraz.com.np,Electronics,5.0,delivery
3,4,Is the third week today since I bought it and its doing great The battery life is way too good Takes about an hour I charged it on day of delivery and didnt need to charge again for a week Has n...,daraz.com.np,Electronics,5.0,product_feedback
4,5,Got the product but left side ko le kaam grdaina i need a return please,daraz.com.np,Electronics,5.0,customer_service


In [46]:
# ── See ALL rows as a scrollable table ────────────────────
pd.set_option('display.max_rows', 100)        # show up to 100 rows at a time
pd.set_option('display.max_colwidth', 200)    # don't truncate review text
pd.set_option('display.width', None)

df

,id,review_text,source,product_category,rating,label
0,1,saama thikai x 2 ota magayeko euta ma problem raixa....euta ramro x.,daraz.com.np,Electronics,5.0,product_feedback
1,2,Thikai xa price aanusar ko lìa hunxa,daraz.com.np,Electronics,5.0,product_feedback
2,3,"nice quality,bass good,I like it time in delivery",daraz.com.np,Electronics,5.0,delivery
3,4,Is the third week today since I bought it and its doing great The battery life is way too good Takes about an hour I charged it on day of delivery and didnt need to charge again for a week Has n...,daraz.com.np,Electronics,5.0,product_feedback
4,5,Got the product but left side ko le kaam grdaina i need a return please,daraz.com.np,Electronics,5.0,customer_service
...,...,...,...,...,...,...
1597,1598,"It has very wide storage area,wow 😊",daraz.com.np,Home Appliances,5.0,product_feedback
1598,1599,thanks for the color I want ♥️ the space is enough for me to manage the mess I just loved it,daraz.com.np,Home Appliances,5.0,product_feedback
1599,1600,Satisfied with the product and delivery.,daraz.com.np,Home Appliances,5.0,delivery
1600,1601,yo stroge bag ykdam rmro lgyra arko choti pani order garyko samn fast delivery grdinu vyko ma thank u daraz teams lai🙏🏿🌸❤️,daraz.com.np,Home Appliances,5.0,delivery


In [47]:
# ── See how many nulls / missing values in each column ────
print('Missing values per column:')
print(df.isnull().sum())
print()
print('Data types:')
print(df.dtypes)

Missing values per column:
id                  0
review_text         0
source              0
product_category    0
rating              0
label               1
dtype: int64

Data types:
id                    int64
review_text             str
source                  str
product_category        str
rating              float64
label                   str
dtype: object


In [48]:
# ── See rating distribution ───────────────────────────────
print('Rating distribution:')
print(df['rating'].value_counts().sort_index())
print()
print('Category distribution:')
print(df['product_category'].value_counts())

Rating distribution:
rating
1.0    303
2.0    214
3.0    235
4.0    260
5.0    590
Name: count, dtype: int64

Category distribution:
product_category
Electronics               639
Health and Beauty         240
Fashion and Clothing      203
Sports & Fitness          145
Home Appliances           127
Stationery                 59
Home & Kitchen             43
Clothing                   18
Beauty                     15
Fashion                    11
Mobile Accessories         11
Beauty & Personal Care     11
Sports & Outdoors          11
Books & Stationery         11
Health & Wellness          11
Groceries                  11
Automotive                 11
Books                       8
Toys                        8
Accessories                 3
Sports                      3
Grocery                     2
Personal Care               1
Name: count, dtype: int64


In [49]:
# ── Drop columns you don't need ───────────────────────────
# Remove any column name from this list if you want to KEEP it
cols_to_drop = [
    'needs_review'
]

# Only drop columns that actually exist in the CSV
cols_to_drop = [c for c in cols_to_drop if c in df.columns]
df = df.drop(columns=cols_to_drop)

print(f'Remaining columns: {list(df.columns)}')
df.head(5)

Remaining columns: ['id', 'review_text', 'source', 'product_category', 'rating', 'label']


,id,review_text,source,product_category,rating,label
0,1,saama thikai x 2 ota magayeko euta ma problem raixa....euta ramro x.,daraz.com.np,Electronics,5.0,product_feedback
1,2,Thikai xa price aanusar ko lìa hunxa,daraz.com.np,Electronics,5.0,product_feedback
2,3,"nice quality,bass good,I like it time in delivery",daraz.com.np,Electronics,5.0,delivery
3,4,Is the third week today since I bought it and its doing great The battery life is way too good Takes about an hour I charged it on day of delivery and didnt need to charge again for a week Has n...,daraz.com.np,Electronics,5.0,product_feedback
4,5,Got the product but left side ko le kaam grdaina i need a return please,daraz.com.np,Electronics,5.0,customer_service


In [50]:
# ── Clean review_text ─────────────────────────────────────
# Strip leading/trailing whitespace
df['review_text'] = df['review_text'].str.strip()

# Collapse multiple spaces/newlines into one space
df['review_text'] = df['review_text'].str.replace(r'\s+', ' ', regex=True)

print('Text cleaning done.')
df[['id', 'review_text', 'rating']].head(10)

Text cleaning done.


,id,review_text,rating
0,1,saama thikai x 2 ota magayeko euta ma problem raixa....euta ramro x.,5.0
1,2,Thikai xa price aanusar ko lìa hunxa,5.0
2,3,"nice quality,bass good,I like it time in delivery",5.0
3,4,Is the third week today since I bought it and its doing great The battery life is way too good Takes about an hour I charged it on day of delivery and didnt need to charge again for a week Has n...,5.0
4,5,Got the product but left side ko le kaam grdaina i need a return please,5.0
5,6,Such a beautiful product with amazing sound ??I love it'??,5.0
6,7,arrived in good condition. i like the way you give extra instruction to remove the plastic cover as it was blocking the bud from charging . the sound quality is also good . satisfied with the product,5.0
7,8,great packaging. time on delivery good colour,5.0
8,9,SABAII WORKING KAI NARAMRO XAINA RAMRO LAGO SOUND BOOM BOOM XA DAMIII LAGO,5.0
9,10,Ramroo nei xaaa sound ni clear aauxa affordable pani ramro panii,5.0


In [51]:
# ── Remove rows where review_text is empty or null ────────
before = len(df)
df = df[df['review_text'].notna()]
df = df[df['review_text'].str.strip() != '']
after = len(df)
print(f'Removed {before - after} empty review rows')
print(f'Remaining: {after} rows')

Removed 0 empty review rows
Remaining: 1602 rows


In [52]:
# ── Remove reviews that are too short to be useful ────────
# Reviews like "good", "ok", "nice" are not useful for intent classification
MIN_CHARS = 7 

before = len(df)
df = df[df['review_text'].str.len() >= MIN_CHARS]
after = len(df)
print(f'Removed {before - after} reviews shorter than {MIN_CHARS} characters')
print(f'Remaining: {after} rows')

# See what the shortest reviews look like now
print('\nShortest reviews remaining:')
df.nsmallest(5, df['review_text'].str.len().name if False else 'id').assign(
    length=df['review_text'].str.len()
).sort_values('length')[['review_text', 'length']].head(5)

Removed 0 reviews shorter than 7 characters
Remaining: 1602 rows

Shortest reviews remaining:


,review_text,length
1,Thikai xa price aanusar ko lìa hunxa,36
2,"nice quality,bass good,I like it time in delivery",49
0,saama thikai x 2 ota magayeko euta ma problem raixa....euta ramro x.,68
4,Got the product but left side ko le kaam grdaina i need a return please,71
3,Is the third week today since I bought it and its doing great The battery life is way too good Takes about an hour I charged it on day of delivery and didnt need to charge again for a week Has n...,314


In [53]:
# ── Remove duplicate review texts ────────────────────────
# (same review text appearing more than once)
before = len(df)
df = df.drop_duplicates(subset='review_text', keep='first')
after = len(df)
print(f'Removed {before - after} duplicate reviews')
print(f'Remaining: {after} rows')

Removed 0 duplicate reviews
Remaining: 1602 rows


In [54]:
# ── Reset the id column after all the cleaning ────────────
df = df.reset_index(drop=True)
df['id'] = df.index + 1

print(f'Final dataset: {len(df)} rows')
df

Final dataset: 1602 rows


,id,review_text,source,product_category,rating,label
0,1,saama thikai x 2 ota magayeko euta ma problem raixa....euta ramro x.,daraz.com.np,Electronics,5.0,product_feedback
1,2,Thikai xa price aanusar ko lìa hunxa,daraz.com.np,Electronics,5.0,product_feedback
2,3,"nice quality,bass good,I like it time in delivery",daraz.com.np,Electronics,5.0,delivery
3,4,Is the third week today since I bought it and its doing great The battery life is way too good Takes about an hour I charged it on day of delivery and didnt need to charge again for a week Has n...,daraz.com.np,Electronics,5.0,product_feedback
4,5,Got the product but left side ko le kaam grdaina i need a return please,daraz.com.np,Electronics,5.0,customer_service
...,...,...,...,...,...,...
1597,1598,"It has very wide storage area,wow 😊",daraz.com.np,Home Appliances,5.0,product_feedback
1598,1599,thanks for the color I want ♥️ the space is enough for me to manage the mess I just loved it,daraz.com.np,Home Appliances,5.0,product_feedback
1599,1600,Satisfied with the product and delivery.,daraz.com.np,Home Appliances,5.0,delivery
1600,1601,yo stroge bag ykdam rmro lgyra arko choti pani order garyko samn fast delivery grdinu vyko ma thank u daraz teams lai🙏🏿🌸❤️,daraz.com.np,Home Appliances,5.0,delivery


In [55]:
# ── Filter and inspect specific rows manually ─────────────
# Change these values to explore your data

# See only 1-star reviews
df[df['rating'] == 1][['id', 'review_text', 'rating', 'product_category']]

,id,review_text,rating,product_category
42,43,I think the worst purchase I ever made in daraz. there is crackling sounds and it keeps disconnecting by itself randomly in like every 5 mins. I tried to reach out to them but they are not even re...,1.0,Electronics
43,44,this bluetooth was not connected with mobile and not bling the light dual side earbirds i want to refund otherwise you have to exchange this thank you,1.0,Electronics
44,45,Name nii dekhaudaina ta connect ta tadai k garne aba,1.0,Electronics
47,48,Seeing review I purchased this product Tara battery full charge garera rakhda ni automatically drain huni ani 30% bata direct 20 %. So I do really want to return this .,1.0,Electronics
49,50,"Maile yo buds kineko 1 mahina mai esko connectivity issue ayo, without any physical damage. Ani maile Greenlife ko customer service ma message gare tesma ni ekdum late response ayo ani seller le 6...",1.0,Electronics
...,...,...,...,...
1575,1576,wrost product ever i want my refunds😠😠 what is these carelessness daraz we donesnt expect this from you,1.0,Fashion and Clothing
1585,1586,"This feels like a scam. They display a nice photo with a good description but deliver a completely different product. Even when I message them saying I need the same product shown in the photo, th...",1.0,Home Appliances
1586,1587,Bad experience buying from this store.I wanted the colour black and they send me this one still I was okay with it because they have exchange policy but what they did is they haven’t reply me alth...,1.0,Home Appliances
1587,1588,eauta bottle order gareko arko botal aaira xa 😰,1.0,Home Appliances


In [56]:
# ── See reviews from a specific category ──────────────────
df[df['product_category'] == 'Electronics'][['id', 'review_text', 'rating']].head(20)

,id,review_text,rating
0,1,saama thikai x 2 ota magayeko euta ma problem raixa....euta ramro x.,5.0
1,2,Thikai xa price aanusar ko lìa hunxa,5.0
2,3,"nice quality,bass good,I like it time in delivery",5.0
3,4,Is the third week today since I bought it and its doing great The battery life is way too good Takes about an hour I charged it on day of delivery and didnt need to charge again for a week Has n...,5.0
4,5,Got the product but left side ko le kaam grdaina i need a return please,5.0
5,6,Such a beautiful product with amazing sound ??I love it'??,5.0
6,7,arrived in good condition. i like the way you give extra instruction to remove the plastic cover as it was blocking the bud from charging . the sound quality is also good . satisfied with the product,5.0
7,8,great packaging. time on delivery good colour,5.0
8,9,SABAII WORKING KAI NARAMRO XAINA RAMRO LAGO SOUND BOOM BOOM XA DAMIII LAGO,5.0
9,10,Ramroo nei xaaa sound ni clear aauxa affordable pani ramro panii,5.0


In [57]:
# ── Search for a keyword in review text ───────────────────
keyword = 'delivery'   # ← change this to whatever you want to search

mask = df['review_text'].str.contains(keyword, case=False, na=False)
results = df[mask][['id', 'review_text', 'rating', 'product_category']]
print(f'Found {len(results)} reviews containing "{keyword}"')
results

Found 178 reviews containing "delivery"


,id,review_text,rating,product_category
2,3,"nice quality,bass good,I like it time in delivery",5.0,Electronics
3,4,Is the third week today since I bought it and its doing great The battery life is way too good Takes about an hour I charged it on day of delivery and didnt need to charge again for a week Has n...,5.0,Electronics
7,8,great packaging. time on delivery good colour,5.0,Electronics
22,23,nice product packing garne tarika dherai ramro lagyo and delivery ni xitai vayo if koi ninne sochma hunu hunxa vane kinda hunxa hai price anusar product ramrai xa ??,4.0,Electronics
35,36,Price 879 and delivery free but after buying it cost 952 i am socked,4.0,Electronics
...,...,...,...,...
1578,1579,As I want it I get it thank you 😊 very much for the delivery very fast thank you hajur lai MVP stores,5.0,Fashion and Clothing
1582,1583,Thank you so much❤️🥰 its perfectly working and its affordable❤️ looks stylish.. Hope it will last.. Very fast shipment and delivery. Will order again❤️,5.0,Home Appliances
1591,1592,i am not satisfied .... photo ma thulo dekhayera delivery chahi sano garxan ... 👿👿👿,5.0,Home Appliances
1599,1600,Satisfied with the product and delivery.,5.0,Home Appliances


In [58]:
# ── Remove reviews containing Nepali (Devanagari) script ─
before = len(df)

nepali_pattern = r'[\u0900-\u097F]'
df = df[~df['review_text'].str.contains(nepali_pattern, regex=True, na=False)]

df = df.reset_index(drop=True)
df['id'] = df.index + 1

after = len(df)
print(f'Removed  : {before - after} reviews with Nepali script')
print(f'Remaining: {after} rows')

Removed  : 0 reviews with Nepali script
Remaining: 1602 rows


In [59]:
# ── Manually delete a specific row by id ──────────────────
# If you see a row that looks wrong, put its id here and run this cell
ids_to_remove = [5, 12, 47]   # ← replace with the actual id numbers you want to remove

df = df[~df['id'].isin(ids_to_remove)]
df = df.reset_index(drop=True)
df['id'] = df.index + 1
print(f'Removed rows with ids: {ids_to_remove}')
print(f'Remaining: {len(df)} rows')

Removed rows with ids: [5, 12, 47]
Remaining: 1599 rows


In [60]:
# ── Final summary before saving ───────────────────────────
print('='*50)
print(f'Final row count     : {len(df)}')
print(f'Columns             : {list(df.columns)}')
print()
print('Rating distribution:')
print(df['rating'].value_counts().sort_index())
print()
print('Category distribution:')
print(df['product_category'].value_counts())
print()
print('Null values:')
print(df.isnull().sum())

Final row count     : 1599
Columns             : ['id', 'review_text', 'source', 'product_category', 'rating', 'label']

Rating distribution:
rating
1.0    303
2.0    213
3.0    235
4.0    260
5.0    588
Name: count, dtype: int64

Category distribution:
product_category
Electronics               636
Health and Beauty         240
Fashion and Clothing      203
Sports & Fitness          145
Home Appliances           127
Stationery                 59
Home & Kitchen             43
Clothing                   18
Beauty                     15
Fashion                    11
Mobile Accessories         11
Beauty & Personal Care     11
Sports & Outdoors          11
Books & Stationery         11
Health & Wellness          11
Groceries                  11
Automotive                 11
Books                       8
Toys                        8
Accessories                 3
Sports                      3
Grocery                     2
Personal Care               1
Name: count, dtype: int64

Null values:

In [ ]:
# ── Save the cleaned CSV ──────────────────────────────────
output_file = 'daraz_reviews_labelled.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f'Saved {len(df)} rows to {output_file}')

Saved 1599 rows to daraz_reviews_labelled_final.csv
